# Scalar Theory Playground

A tiny notebook for poking at the current `pychete` API. It builds a light scalar `phi^4` theory, prints the Lagrangian and EOM in LaTeX, shows noncommutative spinor chains, then integrates out heavy fields at tree level.

Run this with the managed environment from a shell where the Symbolica license is available, for example:

```sh
source "$HOME/.bashrc"
source dependencies/.venv/bin/activate
```

If you launch Jupyter from that environment, the displayed equations below should render as LaTeX.

In [ ]:
from pathlib import Path
import re
import sys

project_root = Path.cwd()
if not (project_root / "src" / "pychete").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

try:
    from IPython.display import Markdown, Math, display
except ImportError:
    class Markdown(str):
        pass

    class Math(str):
        pass

    def display(value):
        print(value)

from symbolica import Expression, PrintMode

from pychete import (
    FieldMassKind,
    Theory,
    canonical_string,
    latex_string,
    relabel_dummy_indices,
    s,
)

FORMAT_OPTIONS = {
    "max_line_length": None,
    "color_top_level_sum": False,
    "color_builtin_symbols": False,
    "bracket_level_colors": None,
    "print_ring": False,
    "multiplication_operator": "*",
    "num_exp_as_superscript": False,
}


def _tidy_imaginary_unit(text, *, latex=False):
    text = re.sub(r"(?<![0-9A-Za-z_.])([+-]?)1𝑖(?=([*/\\\s]|$))", r"\1𝑖", text)
    return text.replace("𝑖", r"\mathrm{i}") if latex else text


def as_symbolica(expr):
    return _tidy_imaginary_unit(expr.format(mode=PrintMode.Symbolica, **FORMAT_OPTIONS))


def as_latex(expr):
    return _tidy_imaginary_unit(latex_string(expr), latex=True)


def is_zero_expression(expr):
    return bool(expr.expand().rationalize().expand() == Expression.num(0))


def show_expr(label, expr, *, canonical=False, show_namespaces=False):
    display(Markdown(f"**{label}**"))
    display(Math(as_latex(expr)))
    print(as_symbolica(expr))
    if canonical:
        print("\ncanonical:")
        print(canonical_string(expr, show_namespaces=show_namespaces))

## 1. A light scalar `phi^4` theory

This is the simplest useful scalar example: a real light scalar with mass `m` and quartic coupling `lambda`.

In [2]:
phi4 = Theory("phi4_playground")
phi = phi4.define_field(
    "phi",
    s.Scalar,
    self_conjugate=True,
    mass=(FieldMassKind.LIGHT, "m"),
)
lam = phi4.define_coupling("lambda", self_conjugate=True)

lagrangian_phi4 = phi4.free_lag(phi) - lam() * phi() ** 4 / 24

display(Markdown("Field and coupling handles render with their notebook hooks:"))
display(phi)
display(lam)

show_expr("phi^4 Lagrangian", lagrangian_phi4)

Field and coupling handles render with their notebook hooks:

**phi^4 Lagrangian**

<IPython.core.display.Math object>

-1/2*phi^2*m^2-1/24*phi^4*lambda+1/2*D[d0](phi)^2


`Theory.derive_eom(...)` gives the Euler-Lagrange equation for a field. The expression below is the left-hand side of the equation of motion.

In [3]:
eom_phi = phi4.derive_eom(lagrangian_phi4, phi)

show_expr("EOM for phi", eom_phi)
show_expr("Mass expression attached to phi", phi4.mass_expr(phi.definition))

**EOM for phi**

<IPython.core.display.Math object>

-phi*m^2-D[d0, d0](phi)-1/6*phi^3*lambda


**Mass expression attached to phi**

<IPython.core.display.Math object>

m


## 2. A heavy scalar matched onto a light theory

Now add a heavy real scalar `S` with mass `M` and interaction `-g S phi^2 / 2`. This is the small tree-level matching example covered by the tests.

In [4]:
heavy_scalar = Theory("heavy_scalar_playground")
S_field = heavy_scalar.define_field(
    "S",
    s.Scalar,
    self_conjugate=True,
    mass=(FieldMassKind.HEAVY, "M"),
)
light_phi = heavy_scalar.define_field("phi", s.Scalar, self_conjugate=True, mass=0)
g = heavy_scalar.define_coupling("g", self_conjugate=True)

lagrangian_heavy = (
    heavy_scalar.free_lag(S_field, light_phi)
    - g() * S_field() * light_phi() ** 2 / 2
)

show_expr("UV Lagrangian", lagrangian_heavy)

**UV Lagrangian**

<IPython.core.display.Math object>

-1/2*S*phi^2*g-1/2*S^2*M^2+1/2*D[d0](S)^2+1/2*D[d0](phi)^2


The heavy-field EOM is solved order by order in the EFT expansion. The first nonzero term is the familiar algebraic source divided by `M^2`; the higher-order term contains derivatives of the light field.

In [5]:
eom_S = heavy_scalar.derive_eom(lagrangian_heavy, S_field)
solution = heavy_scalar.solve_heavy_scalar_eoms(lagrangian_heavy, eft_order=6)["S"]

show_expr("EOM for S", eom_S)

for order in (1, 2, 3):
    show_expr(f"S solution, EFT order {order}", relabel_dummy_indices(solution.orders[order]))

**EOM for S**

<IPython.core.display.Math object>

-S*M^2-D[d0, d0](S)-1/2*phi^2*g


**S solution, EFT order 1**

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

0


**S solution, EFT order 3**

<IPython.core.display.Math object>

phi*D[d0, d0](phi)*g/M^4+D[d0](phi)^2*g/M^4


Finally substitute the heavy-field solution back into the theory to get the matched light-field Lagrangian through dimension six.

In [6]:
matched_lagrangian = heavy_scalar.match(lagrangian_heavy, eft_order=6)

show_expr(
    "Matched light-field Lagrangian through dimension six",
    matched_lagrangian,
    canonical=True,
)

**Matched light-field Lagrangian through dimension six**

<IPython.core.display.Math object>

1/2*phi^2*D[d0](phi)^2*g^2/M^4+1/8*phi^4*g^2/M^2+1/2*D[d0](phi)^2

canonical:
1/2*pychete::Field(heavy_scalar_playground::field_phi,pychete::Scalar,pychete::InternalIndices(),pychete::DerivativeIndices())^2*pychete::Field(heavy_scalar_playground::field_phi,pychete::Scalar,pychete::InternalIndices(),pychete::DerivativeIndices(pychete::Index(pychete::dummy_index(0),pychete::Lorentz)))^2*pychete::Coupling(heavy_scalar_playground::coupling_g,pychete::InternalIndices(),0)^2/pychete::Coupling(heavy_scalar_playground::coupling_M,pychete::InternalIndices(),0)^4+1/8*pychete::Field(heavy_scalar_playground::field_phi,pychete::Scalar,pychete::InternalIndices(),pychete::DerivativeIndices())^4*pychete::Coupling(heavy_scalar_playground::coupling_g,pychete::InternalIndices(),0)^2/pychete::Coupling(heavy_scalar_playground::coupling_M,pychete::InternalIndices(),0)^2+1/2*pychete::Field(heavy_scalar_playground::field_phi,pychete::Scalar,pychete::InternalIndices(),pychete::DerivativeIndices(pychete::Index(

## 3. Noncommutative spinor chains

`ncm_expr(...)` accepts raw Dirac-space matrix atoms as convenient input. During normalization, consecutive `Gamma`, `PL`, and `PR` objects are collected into one canonical `DiracProduct(...)` segment inside the surrounding spinor `NCM`.

In [7]:
from pychete.spinor import ncm_expr, spin_chain_kind

spinor_demo = Theory("ncm_playground")
psi_ncm = spinor_demo.define_field("psi", s.Fermion, mass=0)
mu_ncm = spinor_demo.lorentz_index("mu")
nu_ncm = spinor_demo.lorentz_index("nu")

closed_chain = ncm_expr(s.Bar(psi_ncm()), s.Gamma(mu_ncm), s.PL, psi_ncm())
matrix_chain = ncm_expr(s.Gamma(mu_ncm), s.Gamma(nu_ncm))

show_expr("Closed spinor chain with raw Gamma and PL input", closed_chain, canonical=True)
print("spin-chain kind:", spin_chain_kind(closed_chain))
print()

show_expr("Pure matrix chain", matrix_chain, canonical=True)
print("spin-chain kind:", spin_chain_kind(matrix_chain))

**Closed spinor chain with raw Gamma and PL input**

<IPython.core.display.Math object>

bar(psi) ** gamma(mu) ** P_L ** psi

canonical:
pychete::NCM(pychete::Bar(pychete::Field(ncm_playground::field_psi,pychete::Fermion,pychete::InternalIndices(),pychete::DerivativeIndices())),pychete::DiracProduct(pychete::Gamma(pychete::Index(python::mu,pychete::Lorentz)),pychete::PL),pychete::Field(ncm_playground::field_psi,pychete::Fermion,pychete::InternalIndices(),pychete::DerivativeIndices()))
spin-chain kind: closed



**Pure matrix chain**

<IPython.core.display.Math object>

gamma(mu) ** gamma(nu)

canonical:
pychete::DiracProduct(pychete::Gamma(pychete::Index(python::mu,pychete::Lorentz)),pychete::Gamma(pychete::Index(python::nu,pychete::Lorentz)))
spin-chain kind: matrix


Projector simplification happens before the matrix segment is wrapped. Repeated Lorentz gamma matrices are kept inside `DiracProduct`; gamma algebra simplification is a separate idenso-backed step, not part of `NCM` normalization.

In [8]:
projector_flip = ncm_expr(s.PL, s.Gamma(mu_ncm))
projector_zero = ncm_expr(s.Bar(psi_ncm()), s.PR, s.PL, psi_ncm())
repeated_gamma = ncm_expr(s.Gamma(mu_ncm), s.Gamma(mu_ncm))

show_expr("PL moved through Gamma(mu)", projector_flip, canonical=True)
show_expr("PR followed by PL collapses to zero", projector_zero, canonical=True)
show_expr("Repeated Lorentz gamma pair stays in DiracProduct", repeated_gamma, canonical=True)

**PL moved through Gamma(mu)**

<IPython.core.display.Math object>

gamma(mu) ** P_R

canonical:
pychete::DiracProduct(pychete::Gamma(pychete::Index(python::mu,pychete::Lorentz)),pychete::PR)


**PR followed by PL collapses to zero**

<IPython.core.display.Math object>

0

canonical:
0


**Repeated Lorentz gamma pair stays in DiracProduct**

<IPython.core.display.Math object>

gamma(mu) ** gamma(mu)

canonical:
pychete::DiracProduct(pychete::Gamma(pychete::Index(python::mu,pychete::Lorentz)),pychete::Gamma(pychete::Index(python::mu,pychete::Lorentz)))


## 4. The VLF toy model at tree level

The bundled VLF toy model can be loaded either from the Python asset or from the small supported Matchete/Wolfram subset. The two inputs should construct the same theory metadata and Lagrangian.


In [9]:
from pychete import FieldVariation, bar_expr
from pychete.loaders import load_matchete_model, load_python_model
from pychete.spinor import ncm_expr, spin_chain_kind
from symbolica import Expression

vlf_python_path = project_root / "assets" / "models" / "VLF_toy_model.py"
vlf_mathematica_path = project_root / "assets" / "models" / "VLF_toy_model.m"

vlf, vlf_expressions = load_python_model(vlf_python_path)
vlf_from_mathematica, vlf_mathematica_expressions = load_matchete_model(vlf_mathematica_path)

print("Python and Mathematica assets define the same theory:", vlf.to_json_obj() == vlf_from_mathematica.to_json_obj())
print("Python and Mathematica assets define the same Lagrangian:", canonical_string(vlf_expressions["lagrangian"]) == canonical_string(vlf_mathematica_expressions["lagrangian"]))
print()
print("Fields")
for name, definition in vlf.fields.items():
    charge_text = ", ".join(as_symbolica(charge) for charge in definition.charge_exprs)
    print(f"  {name:>3}: type={as_symbolica(definition.type_expr):<18} heavy={str(bool(definition.heavy)):<5} charges=[{charge_text}]")

show_expr("VLF UV Lagrangian", vlf_expressions["lagrangian"])


Python and Mathematica assets define the same theory: True
Python and Mathematica assets define the same Lagrangian: True

Fields
    A: type=Vector(U1e)        heavy=False charges=[]
  Psi: type=Fermion            heavy=True  charges=[U1e(1)]
  psi: type=Fermion            heavy=False charges=[U1e(1)]
  phi: type=Scalar             heavy=False charges=[]


**VLF UV Lagrangian**

<IPython.core.display.Math object>

-phi*bar(y)*bar(Psi) ** P_L ** psi-phi*bar(psi) ** P_R ** Psi*y-1/2*phi^2*m^2+1/2*D[d0](phi)^2-1/4*F[A, d0, d1]^2/e^2-bar(Psi) ** Psi*M+𝑖*bar(Psi) ** gamma(d0) ** D[d0](Psi)+𝑖*bar(psi) ** gamma(d0) ** D[d0](psi)


The heavy vector-like fermion EOM is an open spin chain after varying with respect to `Bar[Psi]`. The first solution is the Yukawa source divided by the heavy mass; the next term is the derivative correction needed for the dimension-six tree result.


In [10]:
Psi = vlf.field_handle("Psi")
psi = vlf.field_handle("psi")
phi_vlf = vlf.field_handle("phi")
y_vlf = vlf.coupling_handle("y")
M_vlf = vlf.coupling_handle("M")

heavy_eom = vlf.derive_eom(vlf_expressions["lagrangian"], Psi, variation=FieldVariation.BAR)
heavy_solution = vlf.solve_heavy_fermion_eoms(vlf_expressions["lagrangian"], eft_order=6)["Psi"]

show_expr("Heavy fermion EOM from variation with respect to Bar[Psi]", heavy_eom)
source_chain = ncm_expr(s.PL, psi())
print("Source spin-chain kind:", spin_chain_kind(source_chain))

for order in (1, 2):
    show_expr(f"Psi solution, EFT order {order}", relabel_dummy_indices(heavy_solution.orders[order]))


**Heavy fermion EOM from variation with respect to Bar[Psi]**

<IPython.core.display.Math object>

-Psi*M-phi*bar(y)*P_L ** psi+𝑖*gamma(d0) ** D[d0](Psi)
Source spin-chain kind: left_open


**Psi solution, EFT order 1**

<IPython.core.display.Math object>

-phi*bar(y)*P_L ** psi/M


**Psi solution, EFT order 2**

<IPython.core.display.Math object>

-𝑖*phi*bar(y)*gamma(d0) ** P_L ** D[d0](psi)/M^2-𝑖*D[d0](phi)*bar(y)*gamma(d0) ** P_L ** psi/M^2


Tree-level matching through dimension six removes the heavy `Psi` field. The final check below compares the result with the raw off-shell Matchete form, before any IBP/Green simplification.


In [11]:
matched_vlf = vlf.match(vlf_expressions["lagrangian"], eft_order=6, loop_order=0)
mu_vlf = vlf.dummy_index(0)
light_vlf_lagrangian = vlf.free_lag("A", "psi", "phi")
raw_tree_operator = (
    Expression.I
    * phi_vlf() ** 2
    * y_vlf()
    * bar_expr(y_vlf())
    * (
        ncm_expr(s.Bar(psi()), s.Gamma(mu_vlf), s.PL, psi(derivatives=[mu_vlf]))
        - ncm_expr(s.Bar(psi(derivatives=[mu_vlf])), s.Gamma(mu_vlf), s.PL, psi())
    )
    / (2 * M_vlf() ** 2)
)
expected_vlf = light_vlf_lagrangian + raw_tree_operator
difference = (matched_vlf - expected_vlf).expand()

show_expr("Matched VLF Lagrangian through dimension six", matched_vlf)
show_expr("Difference from raw off-shell target", difference)

assert is_zero_expression(difference), as_symbolica(difference)
assert "field_Psi" not in canonical_string(matched_vlf)
print("Heavy Psi removed and raw off-shell target matched.")


**Matched VLF Lagrangian through dimension six**

<IPython.core.display.Math object>

𝑖/2*phi^2*bar(y)*bar(psi) ** gamma(d0) ** P_L ** D[d0](psi)*y/M^2-𝑖/2*phi^2*bar(y)*bar(D[d0](psi)) ** gamma(d0) ** P_L ** psi*y/M^2-1/2*phi^2*m^2+1/2*D[d0](phi)^2-1/4*F[A, d0, d1]^2/e^2+𝑖*bar(psi) ** gamma(d0) ** D[d0](psi)


**Difference from raw off-shell target**

<IPython.core.display.Math object>

0
Heavy Psi removed and raw off-shell target matched.
